In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/anrf-aise-hack-2026-round-1-sar-crop-mapping-challenge/Sample_submission_file.csv
/kaggle/input/competitions/anrf-aise-hack-2026-round-1-sar-crop-mapping-challenge/CAPELLA_C14_SM_SLC_HH_20250606072501_20250606072506/CAPELLA_C14_SM_SLC_HH_20250606072501_20250606072506.json
/kaggle/input/competitions/anrf-aise-hack-2026-round-1-sar-crop-mapping-challenge/CAPELLA_C14_SM_SLC_HH_20250606072501_20250606072506/CAPELLA_C14_SM_SLC_HH_20250606072501_20250606072506.tif
/kaggle/input/competitions/anrf-aise-hack-2026-round-1-sar-crop-mapping-challenge/CAPELLA_C14_SM_SLC_HH_20250606072501_20250606072506/CAPELLA_C14_SM_SLC_HH_20250606072501_20250606072506_extended.json
/kaggle/input/competitions/anrf-aise-hack-2026-round-1-sar-crop-mapping-challenge/CAPELLA_C14_SM_SLC_HH_20250606072501_20250606072506/CAPELLA_C14_SM_SLC_HH_20250606072501_20250606072506_digest.json
/kaggle/input/competitions/anrf-aise-hack-2026-round-1-sar-crop-mapping-challenge/CAPELLA_C14_SM_SLC_HH_20250606

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from sklearn.decomposition import NMF
from sklearn.preprocessing import MinMaxScaler
import warnings

warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. DYNAMIC DIRECTORY SETUP & FILE DISCOVERY
# ---------------------------------------------------------
print("--- Step 1: Discovering Files Dynamically ---")
INPUT_DIR = '/kaggle/input'
WORKING_DIR = '/kaggle/working'

# Find the Sample Submission File
sample_sub_paths = glob.glob(os.path.join(INPUT_DIR, '**/*sample_sub*.csv'), recursive=True) + \
                   glob.glob(os.path.join(INPUT_DIR, '**/*Sample_submission*.csv'), recursive=True)
if not sample_sub_paths:
    raise FileNotFoundError("Could not find the sample submission CSV in the input directory.")
SAMPLE_SUB_PATH = sample_sub_paths[0]

# Find the Village Shapefile
shp_paths = glob.glob(os.path.join(INPUT_DIR, '**/*.shp'), recursive=True)
if not shp_paths:
    raise FileNotFoundError("Could not find a .shp file in the input directory.")
SHP_PATH = shp_paths[0]

# Find Geocoded (GEO) SAR images
all_tifs = glob.glob(os.path.join(INPUT_DIR, '**/*.tif'), recursive=True)
sar_images = sorted([f for f in all_tifs if 'GEO' in f.upper()])
if not sar_images:
    print("Warning: No GEO images found. Falling back to all .tif files.")
    sar_images = sorted(all_tifs)

print(f"Sample Submission: {os.path.basename(SAMPLE_SUB_PATH)}")
print(f"Shapefile: {os.path.basename(SHP_PATH)}")
print(f"Found {len(sar_images)} geocoded SAR images to process.")

# ---------------------------------------------------------
# 2. DYNAMIC CROP TARGETS & GEOMETRY
# ---------------------------------------------------------
print("\n--- Step 2: Extracting Dynamic Targets & Geometries ---")
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

# Dynamically determine the ID column and the crop columns
id_col = sample_sub.columns[0]
crop_cols = [col for col in sample_sub.columns if col != id_col]
n_crops = len(crop_cols)
print(f"Target Crops ({n_crops}): {crop_cols}")

gdf = gpd.read_file(SHP_PATH)

# Ensure the shapefile has the matching ID column
if id_col not in gdf.columns:
    gdf[id_col] = sample_sub[id_col].values

# DYNAMIC CRS / EPSG CALCULATION FOR ACCURATE AREA
# Calculate the exact local UTM zone based on the shapefile centroid for precise hectare calculation
if gdf.crs is not None and gdf.crs.is_geographic:
    centroid = gdf.to_crs(epsg=4326).geometry.unary_union.centroid
    utm_zone = int((centroid.x + 180) / 6) + 1
    dynamic_epsg = 32600 + utm_zone # 32600 represents Northern Hemisphere UTM zones
    
    print(f"Dynamically determined optimal UTM EPSG for accurate area: {dynamic_epsg}")
    gdf_proj = gdf.to_crs(epsg=dynamic_epsg)
    gdf['Village_Area_ha'] = gdf_proj.geometry.area / 10000
else:
    gdf['Village_Area_ha'] = gdf.geometry.area / 10000

# ---------------------------------------------------------
# 3. ZONAL STATISTICS (SAR FEATURE EXTRACTION)
# ---------------------------------------------------------
print("\n--- Step 3: Extracting SAR Zonal Statistics ---")
village_features = []

for idx, tif_path in enumerate(sar_images):
    print(f"Processing Image {idx+1}/{len(sar_images)}: {os.path.basename(tif_path)}")
    with rasterio.open(tif_path) as src:
        if src.crs is None:
            print(f"  -> Skipping {os.path.basename(tif_path)}: No CRS found.")
            continue
            
        gdf_matched = gdf.to_crs(src.crs)
        means = []
        for geom in gdf_matched.geometry:
            try:
                out_image, _ = mask(src, [geom], crop=True, filled=False)
                out_image = np.ma.masked_less_equal(out_image, 0)
                if out_image.count() > 0:
                    # Applied inline mathematical stability constant (1e-10) to prevent log(0)
                    db_val = np.nanmean(10 * np.log10(out_image.data + 1e-10))
                    means.append(db_val)
                else:
                    means.append(np.nan)
            except Exception:
                means.append(np.nan)
                
        village_features.append(means)

if len(village_features) == 0:
    raise ValueError("No SAR features could be extracted.")

X_features = np.array(village_features).T

# Handle missing values by dynamic column mean imputation
col_means = np.nanmean(X_features, axis=0)
for i in range(X_features.shape[1]):
    if np.isnan(col_means[i]):
        X_features[:, i] = 0  
    else:
        X_features[np.isnan(X_features[:, i]), i] = col_means[i]

print(f"Feature extraction complete. Matrix shape: {X_features.shape}")

# ---------------------------------------------------------
# 4. DYNAMIC UNSUPERVISED MODELING
# ---------------------------------------------------------
print("\n--- Step 4: Unsupervised Crop Area Modeling ---")

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_features)

# Automatically choose the right initialization method based on feature count
n_features = X_scaled.shape[1]
init_method = 'nndsvd' if n_crops <= n_features else 'random'
print(f"Using NMF init method: '{init_method}' (Components: {n_crops}, Features: {n_features})")

nmf = NMF(n_components=n_crops, random_state=42, init=init_method, max_iter=1000)
W = nmf.fit_transform(X_scaled)

# Convert component weights into proportional area allocations
row_sums = W.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1 # Prevent division by zero
W_props = W / row_sums

# --- DYNAMIC ARABLE LAND CALCULATION ---
# Instead of hardcoding 70%, we calculate the temporal standard deviation for each village.
# High variance = High agricultural activity (crops growing/harvesting).
# Low variance = Static land (urban, barren, water).
temporal_std = np.std(X_features, axis=1)

# Normalize against the 95th percentile to establish the dynamic ratio (clipped between 5% and 100%)
max_std = np.percentile(temporal_std, 95)
dynamic_arable_ratio = np.clip(temporal_std / max_std, 0.05, 1.0)

# Scale proportions by the dynamically derived agricultural area of each specific village
predicted_areas = W_props * (gdf['Village_Area_ha'].values * dynamic_arable_ratio)[:, np.newaxis]

# ---------------------------------------------------------
# 5. GENERATE SUBMISSION
# ---------------------------------------------------------
print("\n--- Step 5: Generating Submission ---")

df_pred = pd.DataFrame(predicted_areas, columns=crop_cols)
df_pred[id_col] = gdf[id_col].values

# Reorder columns
final_cols = [id_col] + crop_cols
df_final = df_pred[final_cols]

# Ensure precise matching of IDs with the sample sub
df_submission = sample_sub[[id_col]].merge(df_final, on=id_col, how='left').fillna(0.0)

# Save the final file using inline output name
output_path = os.path.join(WORKING_DIR, '12345.csv')
df_submission.to_csv(output_path, index=False)

print(f"SUCCESS! Pipeline finished. File saved to: {output_path}")
print(df_submission.head())

--- Step 1: Discovering Files Dynamically ---
Sample Submission: Sample_submission_file.csv
Shapefile: villages_clean.shp
Found 4 geocoded SAR images to process.

--- Step 2: Extracting Dynamic Targets & Geometries ---
Target Crops (5): ['Rice_ha', 'Cotton_ha', 'Maize_ha', 'Bajra_ha', 'Groundnut_ha']
Dynamically determined optimal UTM EPSG for accurate area: 32643

--- Step 3: Extracting SAR Zonal Statistics ---
Processing Image 1/4: CAPELLA_C14_SM_GEO_HH_20250606072501_20250606072506_preview.tif
Processing Image 2/4: CAPELLA_C14_SM_GEO_HH_20250619021410_20250619021415_preview.tif
Processing Image 3/4: CAPELLA_C14_SM_GEO_HH_20250814031124_20250814031129_preview.tif
Processing Image 4/4: CAPELLA_C14_SM_GEO_HH_20251013022643_20251013022648_preview.tif
Feature extraction complete. Matrix shape: (29, 4)

--- Step 4: Unsupervised Crop Area Modeling ---
Using NMF init method: 'random' (Components: 5, Features: 4)

--- Step 5: Generating Submission ---
SUCCESS! Pipeline finished. File saved t